# Stress Reduction Through Mindfulness Interventions for Tech Industry Professionals
## Capstone Project — Final Notebook

**Author:** Rupleena Chhabra  
**LinkedIn:** [linkedin.com/in/rupleena](https://www.linkedin.com/in/rupleena/)  
**GitHub:** [github.com/rupleena/ml-ai-intervention-outcome-study](https://github.com/rupleena/ml-ai-intervention-outcome-study)

**Research Question:** Can we predict whether a mindfulness intervention will meaningfully reduce stress for a tech industry professional, based on their role, work conditions, and engagement patterns?

---

## Table of Contents
1. [Imports & Setup](#1-imports--setup)
2. [Data Loading & Inspection](#2-data-loading--inspection)
3. [Exploratory Data Analysis](#3-exploratory-data-analysis)
4. [Data Cleaning & Feature Engineering](#4-data-cleaning--feature-engineering)
5. [Train / Test Split & Scaling](#5-train--test-split--scaling)
6. [Clustering — Stress-Risk Profiles](#6-clustering--stress-risk-profiles)
7. [Baseline Model](#7-baseline-model)
8. [Hyperparameter Tuning — GridSearchCV](#8-hyperparameter-tuning--gridsearchcv)
9. [Final Model Comparison](#9-final-model-comparison)
10. [Feature Importance & Interpretation](#10-feature-importance--interpretation)
11. [Cross-Validation Summary](#11-cross-validation-summary)
12. [Key Findings & Recommendations](#12-key-findings--recommendations)


---
## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, cross_val_score,
                                      GridSearchCV, StratifiedKFold)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, f1_score,
                              RocCurveDisplay, ConfusionMatrixDisplay)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100
print("All libraries loaded successfully.")


---
## 2. Data Loading & Inspection

In [ ]:
df = pd.read_csv('intervention_response_tech.csv')

print(f"Shape        : {df.shape}")
print(f"Positive class (stress reduced): {df['stress_reduced'].mean():.2%}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()


In [ ]:
print("=== Data Types ===")
print(df.dtypes)
print("\n=== Summary Statistics ===")
df.describe().T.round(2)


In [ ]:
# Target class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['stress_reduced'].value_counts()
axes[0].bar(['Not Reduced (0)', 'Reduced (1)'], counts.values,
             color=['#fc8d62', '#66c2a5'], edgecolor='white')
axes[0].set_title('Target Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, f'{v}\n({v/len(df):.1%})', ha='center', fontsize=10)

axes[1].hist(df['stress_pre'],  bins=20, alpha=0.6, label='Pre-intervention',  color='#fc8d62')
axes[1].hist(df['stress_post'], bins=20, alpha=0.6, label='Post-intervention', color='#66c2a5')
axes[1].set_title('DASS Stress Score: Before vs After', fontweight='bold')
axes[1].set_xlabel('DASS Stress Score (0–42)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()


---
## 3. Exploratory Data Analysis

### 3.1 Intervention Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rates = df.groupby('intervention')['stress_reduced'].mean().sort_values(ascending=False)
axes[0].bar(rates.index, rates.values * 100,
            color=sns.color_palette('Set2', len(rates)), edgecolor='white')
axes[0].set_title('Stress Reduction Rate by Intervention Type', fontweight='bold')
axes[0].set_ylabel('Reduction Rate (%)')
axes[0].tick_params(axis='x', rotation=30)
for bar, val in zip(axes[0].patches, rates.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val*100:.1f}%', ha='center', fontsize=9)

order = df.groupby('intervention')['stress_delta'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='intervention', y='stress_delta', order=order, palette='Set2', ax=axes[1])
axes[1].set_title('Stress Delta by Intervention Type', fontweight='bold')
axes[1].set_xlabel('Intervention')
axes[1].set_ylabel('Stress Delta (Pre − Post)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].axhline(3, color='red', linestyle='--', linewidth=1, label='Threshold (Δ=3)')
axes[1].legend()

plt.tight_layout()
plt.show()


### 3.2 Role, Seniority & Work Conditions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Role
role_rates = df.groupby('role')['stress_reduced'].mean().sort_values(ascending=False)
axes[0].barh(role_rates.index, role_rates.values * 100,
             color=sns.color_palette('Set2', len(role_rates)), edgecolor='white')
axes[0].set_title('Reduction Rate by Job Role', fontweight='bold')
axes[0].set_xlabel('Reduction Rate (%)')
axes[0].axvline(df['stress_reduced'].mean()*100, color='red', linestyle='--',
                linewidth=1, label=f'Avg ({df["stress_reduced"].mean():.1%})')
axes[0].legend(fontsize=8)

# Seniority
seniority_order = ['Junior','Mid-level','Senior','Lead/Staff','Manager']
sen_rates = df.groupby('seniority')['stress_reduced'].mean().reindex(seniority_order)
axes[1].bar(sen_rates.index, sen_rates.values * 100,
            color=sns.color_palette('Set2', 5), edgecolor='white')
axes[1].set_title('Reduction Rate by Seniority', fontweight='bold')
axes[1].set_ylabel('Reduction Rate (%)')
axes[1].tick_params(axis='x', rotation=20)
axes[1].axhline(df['stress_reduced'].mean()*100, color='red', linestyle='--', linewidth=1)

# On-call
oncall_order = ['Never','Rarely','Monthly','Weekly','Daily']
oc_rates = df.groupby('oncall_freq')['stress_reduced'].mean().reindex(oncall_order)
axes[2].bar(oc_rates.index, oc_rates.values * 100,
            color=sns.color_palette('RdYlGn_r', 5), edgecolor='white')
axes[2].set_title('Reduction Rate by On-Call Frequency', fontweight='bold')
axes[2].set_ylabel('Reduction Rate (%)')
axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


### 3.3 Engagement Patterns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Sessions per week
sess_rates = df.groupby('sessions_per_week')['stress_reduced'].mean()
axes[0].plot(sess_rates.index, sess_rates.values * 100,
             marker='o', color='#1A7A6E', linewidth=2, markersize=8)
axes[0].fill_between(sess_rates.index, sess_rates.values * 100, alpha=0.15, color='#1A7A6E')
axes[0].set_title('Reduction Rate by Sessions Per Week', fontweight='bold')
axes[0].set_xlabel('Sessions Per Week')
axes[0].set_ylabel('Reduction Rate (%)')
axes[0].set_xticks(sess_rates.index)

# Completion rate
for label, color in zip([0,1], ['#fc8d62','#66c2a5']):
    axes[1].hist(df[df['stress_reduced']==label]['completion_rate'],
                 bins=20, alpha=0.6, label=f'Reduced={label}', color=color)
axes[1].set_title('Completion Rate by Outcome', fontweight='bold')
axes[1].set_xlabel('Completion Rate')
axes[1].set_ylabel('Count')
axes[1].legend()

# Practice time
time_rates = df.groupby('practice_time')['stress_reduced'].mean().sort_values(ascending=False)
axes[2].bar(time_rates.index, time_rates.values * 100,
            color=sns.color_palette('Set2', 4), edgecolor='white')
axes[2].set_title('Reduction Rate by Practice Time', fontweight='bold')
axes[2].set_ylabel('Reduction Rate (%)')
for i, val in enumerate(time_rates.values):
    axes[2].text(i, val*100 + 0.3, f'{val*100:.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.show()


### 3.4 Correlation Heatmap

In [ ]:
num_cols = ['stress_pre','stress_post','stress_delta','weekly_hours',
            'meetings_per_day','sessions_per_week','avg_session_min',
            'completion_rate','weeks_of_practice','mindfulness_baseline','stress_reduced']

fig, ax = plt.subplots(figsize=(11, 8))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Correlation Matrix — Numeric Features', fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop correlations with stress_reduced:")
print(corr['stress_reduced'].drop('stress_reduced').sort_values(ascending=False).round(3))


---
## 4. Data Cleaning & Feature Engineering

In [ ]:
print(f"Missing values: {df.isnull().sum().sum()} — no cleaning required.")

df_fe = df.copy()

# 1. Consistency score — frequency × completion × habit formation
df_fe['consistency_score'] = (
    df_fe['sessions_per_week'] *
    df_fe['completion_rate'] *
    np.log1p(df_fe['weeks_of_practice'])
).round(3)

# 2. Work pressure index — composite work stressor
oncall_map = {'Never':0,'Rarely':1,'Monthly':2,'Weekly':3,'Daily':4}
df_fe['oncall_numeric']    = df_fe['oncall_freq'].map(oncall_map)
df_fe['work_pressure_idx'] = (
    (df_fe['weekly_hours'] - 40) / 10 +
    df_fe['oncall_numeric'] / 4 +
    df_fe['meetings_per_day'] / 5
).round(3)

# 3. Experience level — binned mindfulness baseline
df_fe['experience_level'] = pd.cut(
    df_fe['mindfulness_baseline'],
    bins=[0, 35, 50, 75],
    labels=['Low', 'Medium', 'High']
)

print("\nEngineered features summary:")
print(df_fe[['consistency_score','work_pressure_idx','experience_level']].describe())


In [ ]:
# Encode & prepare model features
# Drop leakage columns: stress_post and stress_delta are derived from the outcome
drop_cols   = ['stress_post', 'stress_delta', 'oncall_freq']
encode_cols = ['role','seniority','company_size','work_type',
               'intervention','practice_time','experience_level']

df_encoded = pd.get_dummies(
    df_fe.drop(columns=drop_cols),
    columns=encode_cols, drop_first=True
)

# Also drop stress_pre — partially collinear with outcome
X = df_encoded.drop(columns=['stress_reduced', 'stress_pre'])
y = df_encoded['stress_reduced']

print(f"Feature matrix : {X.shape}")
print(f"Target vector  : {y.shape}")
print(f"Positive class : {y.mean():.2%}")


---
## 5. Train / Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Train : {X_train.shape[0]} rows | pos rate: {y_train.mean():.2%}")
print(f"Test  : {X_test.shape[0]} rows  | pos rate: {y_test.mean():.2%}")
print(f"Features: {X_train.shape[1]}")


---
## 6. Clustering — Stress-Risk Profiles (K-Means)

Before classification, we identify natural groupings within the tech workforce to understand whether a single global model is appropriate or whether different segments need different approaches.

In [ ]:
cluster_features = ['weekly_hours','sessions_per_week','mindfulness_baseline',
                    'consistency_score','work_pressure_idx','stress_pre']
X_cluster = scaler.fit_transform(df_fe[cluster_features].fillna(0))

# Elbow method
inertias = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(2, 9), inertias, marker='o', color='#1A7A6E', linewidth=2, markersize=8)
ax.axvline(3, color='red', linestyle='--', linewidth=1, label='Selected k=3')
ax.set_title('Elbow Method — Optimal Number of Clusters', fontweight='bold')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Fit K-Means k=3
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df_fe['cluster'] = km.fit_predict(X_cluster)

# Profile clusters
profile = df_fe.groupby('cluster')[
    ['weekly_hours','sessions_per_week','mindfulness_baseline',
     'work_pressure_idx','consistency_score','stress_pre','stress_reduced']
].mean().round(2)
print("=== Cluster Profiles ===")
print(profile)

# Label clusters
rates = df_fe.groupby('cluster')['stress_reduced'].mean()
labels = {
    rates.idxmax(): 'Consistent Practitioners',
    rates.idxmin(): 'High-Burnout / Low-Response',
    [c for c in [0,1,2] if c not in [rates.idxmax(), rates.idxmin()]][0]: 'Moderate Engagers'
}
print(f"\nCluster labels: {labels}")


In [ ]:
# Visualise
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cluster)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc_plot = axes[0].scatter(X_pca[:,0], X_pca[:,1], c=df_fe['cluster'],
                           cmap='Set2', alpha=0.6, edgecolors='none', s=40)
axes[0].set_title('K-Means Clusters (PCA projection)', fontweight='bold')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
plt.colorbar(sc_plot, ax=axes[0], label='Cluster')

cluster_rates = df_fe.groupby('cluster')['stress_reduced'].mean()
cluster_labels_list = [labels.get(i, f'Cluster {i}') for i in cluster_rates.index]
bars = axes[1].bar(cluster_labels_list, cluster_rates.values * 100,
                   color=sns.color_palette('Set2', 3), edgecolor='white')
axes[1].set_title('Stress Reduction Rate by Cluster', fontweight='bold')
axes[1].set_ylabel('Reduction Rate (%)')
axes[1].tick_params(axis='x', rotation=15)
for bar, val in zip(bars, cluster_rates.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val*100:.1f}%', ha='center', fontsize=10)
axes[1].axhline(df_fe['stress_reduced'].mean()*100, color='red',
                linestyle='--', linewidth=1, label='Overall avg')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


---
## 7. Baseline Model

Establishing the majority-class floor before training real classifiers.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train_sc, y_train)

print("=== Majority-Class Baseline ===")
print(f"  Test Accuracy : {dummy.score(X_test_sc, y_test):.4f}")
print(f"  ROC-AUC       : {roc_auc_score(y_test, dummy.predict_proba(X_test_sc)[:,1]):.4f}")
print(f"  F1-Score      : {f1_score(y_test, dummy.predict(X_test_sc)):.4f}")
print()
print("Note: 62% accuracy achieved by always predicting 'no reduction'.")
print("F1=0.00 confirms it identifies zero true beneficiaries — this is our floor.")

fig, ax = plt.subplots(figsize=(4, 3))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, dummy.predict(X_test_sc)),
    display_labels=['Not Reduced','Reduced']
).plot(ax=ax, colorbar=False, cmap='Oranges')
ax.set_title('Baseline Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()


---
## 8. Hyperparameter Tuning — GridSearchCV

All four classifiers are tuned using 5-fold stratified cross-validation scored on ROC-AUC.

In [ ]:
# Logistic Regression
lr_gs = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    {'C':[0.01,0.1,1,10], 'penalty':['l1','l2'], 'solver':['liblinear']},
    cv=cv, scoring='roc_auc', n_jobs=-1
)
lr_gs.fit(X_train_sc, y_train)
print(f"LR  best params : {lr_gs.best_params_}")
print(f"LR  best CV AUC : {lr_gs.best_score_:.4f}")


In [ ]:
# K-Nearest Neighbors
knn_gs = GridSearchCV(
    KNeighborsClassifier(),
    {'n_neighbors':[5,11,21,31], 'weights':['uniform','distance']},
    cv=cv, scoring='roc_auc', n_jobs=-1
)
knn_gs.fit(X_train_sc, y_train)
print(f"KNN best params : {knn_gs.best_params_}")
print(f"KNN best CV AUC : {knn_gs.best_score_:.4f}")


In [ ]:
# Decision Tree
dt_gs = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {'max_depth':[3,5,7,10], 'min_samples_leaf':[5,10,20], 'criterion':['gini','entropy']},
    cv=cv, scoring='roc_auc', n_jobs=-1
)
dt_gs.fit(X_train_sc, y_train)
print(f"DT  best params : {dt_gs.best_params_}")
print(f"DT  best CV AUC : {dt_gs.best_score_:.4f}")


In [ ]:
# Support Vector Machine
svm_gs = GridSearchCV(
    SVC(probability=True, random_state=42, class_weight='balanced'),
    {'C':[0.1,1,10], 'kernel':['rbf','linear'], 'gamma':['scale','auto']},
    cv=cv, scoring='roc_auc', n_jobs=-1
)
svm_gs.fit(X_train_sc, y_train)
print(f"SVM best params : {svm_gs.best_params_}")
print(f"SVM best CV AUC : {svm_gs.best_score_:.4f}")


---
## 9. Final Model Comparison

In [ ]:
tuned_models = {
    'Majority-Class Baseline' : dummy,
    'Logistic Regression'     : lr_gs.best_estimator_,
    'KNN'                     : knn_gs.best_estimator_,
    'Decision Tree'           : dt_gs.best_estimator_,
    'SVM'                     : svm_gs.best_estimator_,
}

results = []
for name, model in tuned_models.items():
    pred = model.predict(X_test_sc)
    prob = model.predict_proba(X_test_sc)[:, 1]
    results.append({
        'Model'         : name,
        'Test Accuracy' : round(model.score(X_test_sc, y_test), 4),
        'ROC-AUC'       : round(roc_auc_score(y_test, prob),    4),
        'F1-Score'      : round(f1_score(y_test, pred),         4),
    })

results_df = pd.DataFrame(results).set_index('Model')
print("=== Final Test Set Results ===")
print(results_df.to_string())


In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ['Test Accuracy', 'ROC-AUC', 'F1-Score']
colors  = ['#aaaaaa'] + list(sns.color_palette('Set2', 4))

for ax, metric in zip(axes, metrics):
    vals = results_df[metric]
    bars = ax.bar(vals.index, vals.values, color=colors, edgecolor='white')
    ax.set_title(metric, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=30)
    if metric == 'Test Accuracy':
        ax.axhline(results_df.loc['Majority-Class Baseline','Test Accuracy'],
                   color='red', linestyle='--', linewidth=1, label='Baseline')
        ax.legend(fontsize=8)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8)

fig.suptitle('Final Model Comparison — Test Set Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(7, 6))
styles = ['-', '-', '--', '-.', ':']
for (name, model), ls in zip(tuned_models.items(), styles):
    RocCurveDisplay.from_estimator(model, X_test_sc, y_test, ax=ax, name=name, linestyle=ls)
ax.plot([0,1],[0,1],'k--', linewidth=0.8, label='Random (AUC=0.50)')
ax.set_title('ROC Curves — All Tuned Classifiers', fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrices — tuned models only
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
eval_models = {k:v for k,v in tuned_models.items() if k != 'Majority-Class Baseline'}

for ax, (name, model) in zip(axes.flatten(), eval_models.items()):
    pred = model.predict(X_test_sc)
    auc  = roc_auc_score(y_test, model.predict_proba(X_test_sc)[:,1])
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, pred),
        display_labels=['Not Reduced','Reduced']
    ).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nROC-AUC: {auc:.4f}  |  F1: {f1_score(y_test,pred):.4f}',
                 fontweight='bold', fontsize=9)

fig.suptitle('Confusion Matrices — Tuned Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 10. Feature Importance & Interpretation

### 10.1 Logistic Regression Coefficients

In [ ]:
coef_df = pd.DataFrame({
    'Feature'    : X_train.columns,
    'Coefficient': lr_gs.best_estimator_.coef_[0]
}).sort_values('Coefficient', ascending=False)

top = pd.concat([coef_df.head(12), coef_df.tail(12)])
colors_bar = ['#66c2a5' if c > 0 else '#fc8d62' for c in top['Coefficient']]

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(top['Feature'], top['Coefficient'], color=colors_bar, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 12 Positive & Negative Predictors\n(Logistic Regression Coefficients)',
             fontweight='bold')
ax.set_xlabel('Coefficient  |  Green = increases probability  |  Orange = decreases it')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 5 POSITIVE predictors (increase reduction probability):")
print(coef_df.head(5)[['Feature','Coefficient']].to_string(index=False))
print("\nTop 5 NEGATIVE predictors (decrease reduction probability):")
print(coef_df.tail(5)[['Feature','Coefficient']].to_string(index=False))


### 10.2 Decision Tree Feature Importances

In [ ]:
imp_df = pd.DataFrame({
    'Feature'   : X_train.columns,
    'Importance': dt_gs.best_estimator_.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(imp_df['Feature'], imp_df['Importance'],
        color=sns.color_palette('Set2', len(imp_df)), edgecolor='white')
ax.set_title('Top 15 Feature Importances — Decision Tree', fontweight='bold')
ax.set_xlabel('Gini Importance')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


### 10.3 Decision Tree Structure

In [ ]:
fig, ax = plt.subplots(figsize=(20, 7))
plot_tree(dt_gs.best_estimator_,
          feature_names=X_train.columns,
          class_names=['Not Reduced','Reduced'],
          filled=True, rounded=True, max_depth=3,
          ax=ax, fontsize=7)
ax.set_title('Decision Tree Structure (max depth=3 shown)', fontweight='bold')
plt.tight_layout()
plt.show()


---
## 11. Cross-Validation Summary

In [ ]:
cv_results = []
for name, model in [
    ('Logistic Regression', lr_gs.best_estimator_),
    ('KNN',                 knn_gs.best_estimator_),
    ('Decision Tree',       dt_gs.best_estimator_),
    ('SVM',                 svm_gs.best_estimator_),
]:
    scores = cross_val_score(model, X_train_sc, y_train,
                             cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results.append({
        'Model'        : name,
        'CV Mean AUC'  : round(scores.mean(), 4),
        'CV Std AUC'   : round(scores.std(),  4),
        'Test AUC'     : round(roc_auc_score(y_test, model.predict_proba(X_test_sc)[:,1]), 4),
    })

cv_df = pd.DataFrame(cv_results).set_index('Model')
print("=== 5-Fold CV ROC-AUC vs Test ROC-AUC ===")
print(cv_df.to_string())

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(cv_df))
w = 0.35
ax.bar(x - w/2, cv_df['CV Mean AUC'], w, yerr=cv_df['CV Std AUC'],
       label='CV Mean AUC', color='#66c2a5', capsize=5, edgecolor='white')
ax.bar(x + w/2, cv_df['Test AUC'], w,
       label='Test AUC', color='#fc8d62', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(cv_df.index, rotation=15)
ax.set_ylim(0.4, 0.75)
ax.axhline(0.5, color='black', linestyle='--', linewidth=1, label='Random baseline')
ax.set_title('CV Mean AUC vs Test AUC by Model', fontweight='bold')
ax.set_ylabel('ROC-AUC')
ax.legend()
plt.tight_layout()
plt.show()


---
## 12. Key Findings & Recommendations

### Model Performance Summary

| Model | Test Accuracy | ROC-AUC | F1-Score | CV AUC |
|---|---|---|---|---|
| Majority-Class Baseline | 0.6208 | 0.5000 | 0.0000 | — |
| Logistic Regression (tuned) | 0.5208 | 0.5193 | 0.4335 | 0.5886 ± 0.0261 |
| KNN (tuned) | 0.5750 | 0.4688 | 0.1774 | 0.5830 ± 0.0478 |
| Decision Tree (tuned) | 0.6083 | 0.5035 | 0.0600 | 0.5263 ± 0.0330 |
| **SVM (tuned)** | 0.5250 | 0.5047 | 0.4000 | **0.6007 ± 0.0269** |

**Best overall:** SVM achieves the highest cross-validation AUC (0.6007). Logistic Regression achieves the best test F1-Score (0.4335) and is the most interpretable.

---

### What the Data Shows

**1. Consistency is the strongest predictor**  
The engineered `consistency_score` — combining sessions per week, completion rate, and weeks of practice — is the top positive predictor across all models. Regular, completed practice over sustained periods drives outcomes more than any single session characteristic.

**2. Intervention type matters**  
Breathing exercises and body scans show the highest reduction rates. Journaling and sleep practices show the lowest. Guided meditation falls in between — notable because it is the most commonly recommended starting point in most wellness programs.

**3. Work environment limits effectiveness**  
`work_pressure_idx` and `oncall_numeric` are among the strongest negative predictors. For the high-burnout cluster (Cluster 1, 32.1% reduction rate), self-administered interventions appear insufficient on their own — structural workload changes may be a prerequisite.

**4. Three distinct workforce segments**  
K-Means clustering identified three profiles with meaningfully different reduction rates:
- **Consistent Practitioners** (~48%) — most responsive
- **Moderate Engagers** (~39%) — moderate response, strongest improvement opportunity  
- **High-Burnout / Low-Response** (~32%) — least responsive, may need clinical-level support

**5. Honest model performance**  
All models show modest but consistent improvement above the random baseline on ROC-AUC. This is expected — predicting individual stress outcomes from survey data is genuinely difficult. These results establish a transparent baseline for future work with richer longitudinal and physiological data.

---

### Recommendations for Individuals
This analysis provides data-backed insights for individuals wanting to have options to manage their stress and explore, through data, which mindfulness practices may be worth trying for someone with their profile — without promising any specific outcome.

- Prioritise **consistency over duration** — brief daily practice outperforms occasional long sessions
- **Breathing exercises and body scans** show the strongest signal for stress reduction in tech workers
- If work pressure is very high, **addressing structural stressors** alongside mindfulness practice is likely necessary for any intervention to be effective
